In [1]:
using Plots, ProgressMeter, Statistics, BSON
include("analysis_tools.jl")

In [2]:
nx=100
L=10.0
ν=0.1
k=2
u_mean=1.0
u_amplitude=0.5
noise_strength=0.001f0
t_end=15
cfl=0.8

X, Δt = burgers_FV(nx, L, ν, k, u_mean, u_amplitude, noise_strength, t_end, cfl);

In [3]:
ks = 0.5:0.1:8.0
amps = 0.1:0.1:2.0
means = [-1.0, 1.0]

samples=5

X = zeros(Float32, nx, 1, samples*length(ks)*length(amps)*length(means))
Δt = zeros(Float32, 1, samples*length(ks)*length(amps)*length(means))
y = zeros(Float32, nx, 1, samples*length(ks)*length(amps)*length(means))

count=1
@showprogress for k in ks
    for u_amplitude in amps
        for u_mean in means
            X_data, Δt_data = burgers_FV(nx, L, ν, k, u_mean, u_amplitude, noise_strength, t_end, cfl)
            y_data, _ = burgers_FV(nx, L, ν, k, u_mean, u_amplitude, 0, t_end, cfl)

            sample_idx = rand(1:length(X_data)-1, samples)
            for t in sample_idx
                X[:,:,count] .= X_data[t]
                Δt[:,count] .= Δt_data

                y[:,:,count] .= y_data[t+1]
                count+=1
            end
        end
    end
end
Xμ, Xσ = mean(X), std(X)
X = (X .- Xμ) ./ Xσ
y = (y .- Xμ) ./ Xσ
println("Amount of data pairs: $(size(X,3))")

Progress: 100%|█████████████████████████████████████████| Time: 0:07:18


Amount of data pairs: 15200


In [4]:
BSON.@save "data/burgers_dataset.bson" X Δt y Xμ Xσ